<a href="https://colab.research.google.com/github/Mv0sKff/KISysteme26_lab/blob/main/block_1/Aufgabe_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 - Matrix multiplication in Numba


We consider the problem of evaluating the matrix multiplication $C = A\times B$ for matrices $A, B\in\mathbb{R}^{n\times n}$.
A simple Python implementation of the matrix-matrix product is given below through the function `matrix_product`. At the end this
function is checked against the Numpy implementation of the matrix-matrix product.

In [1]:
import numpy as np

def matrix_product(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in range(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

a = np.random.randn(10, 10)
b = np.random.randn(10, 10)

c_actual = matrix_product(a, b)
c_expected = a @ b

error = np.linalg.norm(c_actual - c_expected) / np.linalg.norm(c_expected)
print(f"The error is {error}.")


The error is 1.0686533131861634e-16.


The matrix product is one of the most fundamental operations on modern computers. Most algorithms eventually make use of this operation. A lot of effort is therefore spent on optimising the matrix product. Vendors provide hardware optimised BLAS (Basis Linear Algebra Subroutines) that provide highly efficient versions of the matrix product. Alternatively, open-source libraries sucha as Openblas provide widely used generic open-source implementations of this operation.

In this assignment we want to learn at the example of matrix-matrix products about the possible speedups offered by Numba, and the effects of cache-efficient programming.

## 1.1 Benchmark
Benchmark the above function against the Numpy dot product for matrix sizes up to 1000. Plot the timing results of the above function against the timing results for the Numpy dot product. You need not benchmark every dimension up to 1000. Figure out what dimensions to use so that you can represent the result without spending too much time waiting for the code to finish. To perform benchmarks you can use the `%timeit` magic command. An example is

    ```
    timeit_result = %timeit -o matrix_product(a, b)
    print(timeit_result.best)
    ```

## 1.2 Optimize
Now optimise the code by using Numba to JIT-compile it. Also, there is lots of scope for parallelisation in the code. You can for example parallelize the outer-most for-loop. Benchmark the JIT-compiled serial code against the JIT-compiled parallel code. Comment on the expected performance on your system against the observed performance.

## 1.3 (Optional) Cache Optimization
Now let us improve Cache efficiency. Notice that in the matrix $B$ we traverse by columns. However, the default storage ordering in Numpy is row-based. Hence, the expression `mat_b[k, col_ind]` jumps in memory by `n` units if we move from $k$ to $k+1$. Run your parallelized JIT-compiled Numba code again. But this time choose a matrix $B$ that is stored in column-major order. To change an array to column major order you can use the command `np.asfortranarray`.






In [ ]:
from tqdm import tqdm

time_naive = []
time_numpy = []

#timeit_result = %timeit -o matrix_product(a, b)
#print(timeit_result.best)


for i in tqdm(range(1, 1000, 50)):
  a = np.random.randn(i, i)
  b = np.random.randn(i, i)

  #c_naive = matrix_product(a, b)
  #c_numpy = a @ b

  timeit_naive_result = %timeit -n 3 -r 5 -q -o matrix_product(a, b)
  timeit_numpy_result = %timeit -n 3 -r 5 -q -o a @ b

  time_naive.append(timeit_naive_result.average)
  time_numpy.append(timeit_numpy_result.average)

display(time_naive)
display(time_numpy)

1.21 µs ± 495 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.78 µs ± 934 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.37 µs ± 627 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.77 µs ± 859 ns per loop (mean ± std. dev. of 7 runs, 10 loops each)
7.99 µs ± 2.1 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
The slowest run took 4.22 times longer than the fastest. This could mean that an intermediate result is being cached.
3.44 µs ± 1.67 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
26.6 µs ± 6.98 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.15 µs ± 1.05 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
47.3 µs ± 3.7 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.17 µs ± 1.12 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
111 µs ± 37.2 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.22 µs ± 1.26 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
164 µs ± 16.1 µs per loop (mean ± std

In [ ]:
import seaborn as sns

sns.plot(time_naive, time_numpy)